## Install Libraries

Runs the online stages of the GRF pipeline (architecture §5.2/§5.3): load the
KB data (`kb.jsonl`) and the single-file KB artifact built by
`source/notebooks/GRF/build_KB.ipynb`, load the gold BEL data — all from Google
Drive — run the configured GRF pipeline, and evaluate candidate retrieval
(R@1/R@5) and end-to-end accuracy.

Sources: `docs/architecture.md`, `paper/reference_abstract.md`,
`source/GRF/README.md`.

Same pin as `source/GRF/requirements.txt`. Colab already ships numpy and tqdm.

In [ ]:
%pip install -q "sentence-transformers>=2.7"

## Import Libraries

Third-party imports only. Project modules (`GRF.*`) are imported at the end of
the repository-setup stage, after the clone.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from dataclasses import asdict
from pathlib import Path

from tqdm.auto import tqdm

print("Python:", sys.version.split()[0])
print("tqdm:", __import__("tqdm").__version__)

## Constants and Configuration

All paths, method parameters and LLM settings live here. The Git repository
and branch are preconfigured. Set `DRIVE_DATA_DIR` once; the Drive paths below derive from it:

- the KB JSONL (`kb.jsonl`) that `build_KB.ipynb` embeds,
- the `.npz` artifact written by `build_KB.ipynb`,
- the gold BEL JSONL file on Drive,
- the JSON results output path on Drive.

`FEEDBACK_TYPES = ()` reproduces the no-GRF baseline; `("def",)`, `("syn",)`,
`("std",)` select single GRF components; `("def", "syn", "std")` is All-in-One
(architecture §6, §7).

In [ ]:
# ---------------------------------------------------------------------------
# Git repository (REQUIRED: replace the placeholder with your repository URL)
# ---------------------------------------------------------------------------
GIT_REPO_URL = "https://github.com/minhhhduc/bio-linking-.git"
GIT_BRANCH = "main"                                             # TODO
REPO_CLONE_DIR = Path("/content/bio_linking")

# ---------------------------------------------------------------------------
# Google Drive inputs and output (REQUIRED: set DRIVE_DATA_DIR below)
# ---------------------------------------------------------------------------
MOUNT_DRIVE = True
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"
DRIVE_DATA_DIR = DRIVE_ROOT / "REPLACE_ME"                      # TODO: set once
KB_JSONL_PATH = DRIVE_DATA_DIR / "kb.jsonl"                     # KB data (input)
KB_ARTIFACT_PATH = DRIVE_DATA_DIR / "grf_kb.npz"                # from build_KB.ipynb
GOLD_DATA_PATH = DRIVE_DATA_DIR / "test.jsonl"                  # gold BEL JSONL
RESULTS_OUTPUT_PATH = DRIVE_DATA_DIR / "grf_results.json"       # evaluation results

# ---------------------------------------------------------------------------
# Encoder (must match the encoder_info stored in the KB artifact)
# ---------------------------------------------------------------------------
ENCODER_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
ENCODER_BATCH_SIZE = 32
DEVICE = "cuda"                # "cuda" (Colab GPU) or "cpu"

# ---------------------------------------------------------------------------
# GRF method parameters (architecture §9)
# ---------------------------------------------------------------------------
FEEDBACK_TYPES = ("def",)       # (), ("def",), ("syn",), ("std",), or all three
N_SYN = 5                       # one of 1, 3, 5, 10
TARGET_KB = "KB"                # KB name used by the `std` instruction
TOP_K_RETRIEVAL = 10
TOP_K_CONCEPT = 5               # 1..10; candidates passed to the re-ranker
FUSION = "vector"               # "vector" (Pipeline 1) or "rrf" (Pipeline 2)
RERANK = "top1"                 # "top1" or "llm"
INDEX_KB = None                 # optional source_kb filter; None = all entries

# ---------------------------------------------------------------------------
# LLM (OpenAI-compatible; failures degrade to baseline, architecture §6)
# ---------------------------------------------------------------------------
LLM_MODEL = "gpt-4o"
LLM_TIMEOUT_S = 30.0
OPENAI_BASE_URL = ""            # empty -> OPENAI_BASE_URL env var / api.openai.com
# Set the key in this runtime (never commit it):
#   import getpass, os; os.environ["OPENAI_API_KEY"] = getpass.getpass()

# ---------------------------------------------------------------------------
# Evaluation / runtime
# ---------------------------------------------------------------------------
MAX_SAMPLES = None              # None = all messages; an int truncates for smoke runs
CACHE_DIR = Path("/content/grf_cache")
HF_HOME = CACHE_DIR / "huggingface"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_HOME))

_START = time.time()


def log(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')} +{time.time() - _START:6.1f}s] {message}", flush=True)


def require_configured(value, name: str):
    text = str(value)
    if "REPLACE_ME" in text:
        raise ValueError(f"{name} is still a placeholder: {text!r}. Set it in the constants cell.")
    return value


log("Configuration loaded.")
print(f"Repository   : {GIT_REPO_URL} ({GIT_BRANCH})")
print(f"KB data      : {KB_JSONL_PATH}")
print(f"KB artifact  : {KB_ARTIFACT_PATH}")
print(f"Gold data    : {GOLD_DATA_PATH}")
print(f"Results out  : {RESULTS_OUTPUT_PATH}")
print(f"GRF settings : feedback={FEEDBACK_TYPES} n_syn={N_SYN} fusion={FUSION} rerank={RERANK}")

## Stage 1 — Clone or Update the GRF Repository

Clone the code repository, add `source/` to `sys.path`, and import the project
modules. Re-running this cell updates the clone instead of re-downloading it.

In [ ]:
repo_url = str(require_configured(GIT_REPO_URL, "GIT_REPO_URL"))


def _git(*args: str) -> None:
    log("git " + " ".join(args))
    try:
        subprocess.run(["git", *args], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"git {args[0]} failed with exit code {exc.returncode}.") from exc


if REPO_CLONE_DIR.exists() and not (REPO_CLONE_DIR / ".git").exists():
    raise RuntimeError(
        f"{REPO_CLONE_DIR} exists but is not a git clone. "
        "Remove it or point REPO_CLONE_DIR elsewhere."
    )

if (REPO_CLONE_DIR / ".git").exists():
    log(f"Repository already present at {REPO_CLONE_DIR}; updating branch {GIT_BRANCH!r}.")
    _git("-C", str(REPO_CLONE_DIR), "fetch", "--depth", "1", "origin", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "checkout", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "pull", "--ff-only", "origin", GIT_BRANCH)
else:
    log(f"Cloning {repo_url} (branch {GIT_BRANCH!r}) into {REPO_CLONE_DIR} ...")
    REPO_CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    _git("clone", "--depth", "1", "--branch", GIT_BRANCH, repo_url, str(REPO_CLONE_DIR))

SOURCE_DIR = REPO_CLONE_DIR / "source"
if not (SOURCE_DIR / "GRF" / "__init__.py").is_file():
    raise FileNotFoundError(
        f"{SOURCE_DIR / 'GRF' / '__init__.py'} is missing; check GIT_BRANCH and the repository layout."
    )
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from GRF.config import PipelineConfig
from GRF.data import load_bel_jsonl
from GRF.encoder import SentenceTransformerEncoder
from GRF.evaluator import evaluate_all
from GRF.feedback import FeedbackGenerator
from GRF.index import VectorIndex
from GRF.kb import load_concepts_jsonl
from GRF.llm import OpenAICompatibleClient
from GRF.pipeline import EntityLinker
from GRF.rerank import LLMReranker
from GRF.schemas import Request

log(f"Repository ready at {REPO_CLONE_DIR}; GRF imported from {SOURCE_DIR}.")

## Stage 2 — Mount Google Drive and Locate Inputs

The KB data (`kb.jsonl`), the KB artifact and the gold data live on Drive, not
in the repository. This stage mounts Drive and checks every configured input
path before any model is loaded.

In [ ]:
if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:  # running outside Colab
        raise RuntimeError(
            "Google Colab is required to mount Drive. Run this notebook in Colab, "
            "or set MOUNT_DRIVE = False only if the inputs are already local to the runtime."
        ) from exc
    drive.mount(str(DRIVE_MOUNT_POINT))

kb_path = Path(require_configured(KB_JSONL_PATH, "KB_JSONL_PATH"))
artifact_path = Path(require_configured(KB_ARTIFACT_PATH, "KB_ARTIFACT_PATH"))
gold_path = Path(require_configured(GOLD_DATA_PATH, "GOLD_DATA_PATH"))
results_path = Path(require_configured(RESULTS_OUTPUT_PATH, "RESULTS_OUTPUT_PATH"))

for label, path in (
    ("KB data", kb_path),
    ("KB artifact", artifact_path),
    ("Gold data", gold_path),
):
    if not path.is_file():
        raise FileNotFoundError(f"{label} not found: {path}")
    log(f"{label}: {path} ({path.stat().st_size / 1024 ** 2:.2f} MB)")

## Stage 3 — Load the KB Artifact and the Encoder

`VectorIndex.load_npz` restores the vectors and metadata from the single file.
The encoder model itself is not stored in the artifact, so this stage loads the
same checkpoint recorded in `encoder_info` and checks the dimensions agree.

In [ ]:
log(f"Loading KB artifact {artifact_path} ...")
index = VectorIndex.load_npz(artifact_path)
log(f"Artifact loaded: {len(index):,} entries x {index.vectors.shape[1]} dims.")

stored_encoder = (index.encoder_info or {}).get("model")
if stored_encoder is None:
    log("WARNING: artifact has no encoder_info; assuming ENCODER_MODEL is the same checkpoint.")
elif stored_encoder != ENCODER_MODEL:
    raise ValueError(
        f"Artifact was built with encoder {stored_encoder!r}, but ENCODER_MODEL is "
        f"{ENCODER_MODEL!r}. Set ENCODER_MODEL to the encoder used by build_KB.ipynb."
    )

if DEVICE == "cuda":
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "DEVICE='cuda' but no CUDA device is available. In Colab: "
            "Runtime > Change runtime type > Hardware accelerator > GPU."
        )
    log(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    log("Running on CPU (DEVICE='cpu').")

log(f"Loading encoder {ENCODER_MODEL!r} on {DEVICE} ...")
encoder = SentenceTransformerEncoder(
    ENCODER_MODEL, batch_size=ENCODER_BATCH_SIZE, device=DEVICE
)
if encoder.dim != index.vectors.shape[1]:
    raise ValueError(
        f"Encoder dimension {encoder.dim} does not match artifact dimension "
        f"{index.vectors.shape[1]}."
    )
log(f"Encoder ready: dim={encoder.dim}.")

## Stage 4 — Load the KB Data from Google Drive

The KB JSONL on Drive is the authoritative source of preferred concept names
used by the LLM re-ranker. It is loaded here and cross-checked against the CUIs
stored in the artifact so a stale `grf_kb.npz` fails loudly instead of mixing
an old index with a new KB.

In [ ]:
log(f"Loading KB data from {kb_path} ...")
started = time.time()
kb_concepts = load_concepts_jsonl(kb_path)
log(f"Loaded {len(kb_concepts):,} KB concepts in {time.time() - started:.1f}s.")
if not kb_concepts:
    raise ValueError(f"{kb_path} produced no KB concepts.")

concept_names = {concept.cui: concept.name for concept in kb_concepts}
artifact_cuis = set(index.cuis)
kb_cuis = set(concept_names)

missing = artifact_cuis - kb_cuis
if missing:
    raise ValueError(
        f"{len(missing):,} CUIs in the KB artifact are missing from {kb_path}; "
        "the artifact and the KB data do not match. Rebuild it with build_KB.ipynb."
    )
extra = kb_cuis - artifact_cuis
if extra:
    log(
        f"WARNING: {len(extra):,} KB CUIs are absent from the artifact "
        "(the artifact may be stale or filtered)."
    )
log(f"Concept names ready: {len(concept_names):,} CUIs from the KB data.")

## Stage 5 — Load and Validate Gold Data

`load_bel_jsonl` validates structure and token spans. Structural problems are
reported as `DataIssue` entries; span issues never drop a sample. The first
valid record is used by the single-mention stage, and all samples feed the
evaluation stage.

In [ ]:
log(f"Loading gold BEL data from {gold_path} ...")
started = time.time()
loaded = load_bel_jsonl(gold_path, strict=False)
samples = list(loaded.samples)
if MAX_SAMPLES is not None:
    if isinstance(MAX_SAMPLES, bool) or not isinstance(MAX_SAMPLES, int) or MAX_SAMPLES < 1:
        raise ValueError("MAX_SAMPLES must be None or a positive integer.")
    samples = samples[:MAX_SAMPLES]

log(
    f"Loaded {len(loaded.samples):,} mention samples "
    f"({len(samples):,} selected) in {time.time() - started:.1f}s."
)
if loaded.issues:
    log(f"WARNING: {len(loaded.issues):,} data issue(s); first: {loaded.issues[0]}")
if not samples:
    raise ValueError(f"{gold_path} produced no usable mention samples.")

## Stage 6 — Configure and Build the GRF Pipeline

`PipelineConfig` is validated on construction (fusion kind, re-rank kind,
`n_syn` in {1, 3, 5, 10}, `top_k_concept` in 1..10). `EntityLinker` wires the
encoder, index, feedback generator and re-ranker. `feedback_types = ()`
degrades to plain dense retrieval; LLM failures degrade to `Q_0` instead of
raising (architecture §6).

Re-run this cell after changing any constant or re-running an upstream cell:
stages 7–9 reuse the `linker` built here.

In [ ]:
config = PipelineConfig(
    encoder=ENCODER_MODEL,
    feedback_types=tuple(FEEDBACK_TYPES),
    n_syn=N_SYN,
    target_kb=TARGET_KB,
    top_k_retrieval=TOP_K_RETRIEVAL,
    top_k_concept=TOP_K_CONCEPT,
    fusion=FUSION,
    rerank=RERANK,
    llm_model=LLM_MODEL,
    index_kb=INDEX_KB,
)

needs_llm = bool(config.feedback_types) or config.rerank == "llm"
llm = None
if needs_llm:
    if not os.environ.get("OPENAI_API_KEY"):
        log(
            "WARNING: OPENAI_API_KEY is not set. LLM feedback/re-rank calls will "
            "fail and the pipeline will degrade to the baseline path."
        )
    log(f"LLM client: model={config.llm_model!r} base_url={OPENAI_BASE_URL or '<default/env>'}")
    llm = OpenAICompatibleClient(
        config.llm_model,
        base_url=OPENAI_BASE_URL or None,
        timeout_s=LLM_TIMEOUT_S,
    )

feedback_generator = FeedbackGenerator(llm) if config.feedback_types else None
reranker = LLMReranker(llm, timeout_s=LLM_TIMEOUT_S) if config.rerank == "llm" else None

if not concept_names:
    raise ValueError("concept_names is empty; run the KB-data stage first.")

linker = EntityLinker(
    index,
    encoder,
    config,
    feedback_generator=feedback_generator,
    reranker=reranker,
    concept_names=concept_names,
)

log(
    f"Pipeline ready: {len(index):,} index entries, {len(concept_names):,} concepts; "
    f"feedback={config.feedback_types or 'none (baseline)'} "
    f"n_syn={config.n_syn} fusion={config.fusion} rerank={config.rerank}."
)

## Stage 7 — Link One Mention (Interactive Check)

Run the full online pipeline on the first gold sample: feedback generation,
query encoding, retrieval, concept aggregation and re-ranking. This is the
place to inspect candidates and degradation messages before a long evaluation.

In [ ]:
sample = samples[0]
log(f"Linking mention {sample.mention!r} ...")
started = time.time()
response = linker.link(
    Request(
        mention=sample.mention,
        context=sample.context,
        feedback_types=config.feedback_types,
        n_syn=config.n_syn,
    )
)
log(
    f"Predicted {response.cui!r} (gold {sample.gold_cui!r}) in "
    f"{time.time() - started:.1f}s."
)
for rank, candidate in enumerate(response.candidates, start=1):
    log(f"  {rank}. {candidate.cui}  score={candidate.score:.4f}  surface={candidate.surface!r}")

failures = response.debug.get("feedback_failures") or []
if failures:
    log(f"Feedback failures (degraded to Q_0): {failures}")
rerank_error = (response.debug.get("rerank") or {}).get("error")
if rerank_error:
    log(f"Re-ranker error (fell back to top-1): {rerank_error}")

## Stage 8 — Evaluate Candidate Retrieval and End-to-End Accuracy

One pipeline pass produces both overall and per-label metrics (R@1, R@5 and
accuracy after re-ranking, architecture §10). `evaluate_all` reports each
completed mention through `progress_callback`; the bar below reflects real
completed work.

In [ ]:
bar = tqdm(total=len(samples), desc="Linking mentions", unit="mention")


def _progress(done: int, total: int) -> None:
    bar.n = done
    bar.refresh()


try:
    started = time.time()
    overall, by_label = evaluate_all(linker, samples, progress_callback=_progress)
    elapsed = time.time() - started
finally:
    bar.close()

log(f"Evaluated {overall.n:,} mentions in {elapsed:.1f}s.")


def _row(label, metrics) -> str:
    return (
        f"{str(label):<16}{metrics.n:>7}"
        f"{100 * metrics.recall_at_1:>10.2f}{100 * metrics.recall_at_5:>10.2f}"
        f"{100 * metrics.accuracy:>10.2f}"
    )


print(f"{'label':<16}{'n':>7}{'R@1 %':>10}{'R@5 %':>10}{'acc %':>10}")
print(_row("ALL", overall))
for label, metrics in by_label.items():
    print(_row(label, metrics))

## Stage 9 — Save Evaluation Results

Write the resolved config, overall metrics, per-label metrics and artifact
provenance to the single configured `RESULTS_OUTPUT_PATH` on Drive, then read it
back to confirm the file is usable.

In [ ]:
if results_path.suffix.lower() != ".json":
    raise ValueError(f"RESULTS_OUTPUT_PATH must end in .json, got {results_path.name!r}.")

payload = {
    "kb_artifact": str(artifact_path),
    "gold_data": str(gold_path),
    "encoder_model": ENCODER_MODEL,
    "encoder_info": index.encoder_info,
    "config": {
        "encoder": config.encoder,
        "feedback_types": list(config.feedback_types),
        "n_syn": config.n_syn,
        "target_kb": config.target_kb,
        "top_k_retrieval": config.top_k_retrieval,
        "top_k_concept": config.top_k_concept,
        "fusion": config.fusion,
        "rerank": config.rerank,
        "llm_model": config.llm_model,
        "index_kb": config.index_kb,
    },
    "n_samples": len(samples),
    "elapsed_s": elapsed,
    "overall": asdict(overall),
    "by_label": {str(label): asdict(metrics) for label, metrics in by_label.items()},
}

results_path.parent.mkdir(parents=True, exist_ok=True)
log(f"Saving results to {results_path} ...")
results_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
saved = json.loads(results_path.read_text(encoding="utf-8"))
assert saved["overall"] == payload["overall"]
log(
    f"Saved and verified {results_path} "
    f"({results_path.stat().st_size / 1024 ** 2:.2f} MB)."
)
log("Run complete.")